# Introduction
This notebook is inspired by the discussion (https://www.kaggle.com/competitions/hubmap-hacking-the-human-vasculature/discussion/419309). 

According to the [competition info](https://www.kaggle.com/competitions/hubmap-hacking-the-human-vasculature/overview/evaluation), the evaluation code is public available.

The idea is to make the validation simple:

- Run your inference pipeline on your validation set and write the results in the submission format.
- Upload the validation output in submission format as a dataset and update the path in this notebook.
- Execute this notebook to get the metrics.

Since the upload-evaluate process may be annoying, I have put notes for how to run the same evaluation locally. 

# Install Dependencies
For local test, you can install the dependencies and comment out this block.

`protobuf-compiler` is required in my ubuntu env to run `protoc`

In [1]:
# Eval Dependencies
!pip install -q /kaggle/input/oid-eval-dependencies/tensorflow_io-0.31.0-cp310-cp310-manylinux_2_12_x86_64.manylinux2010_x86_64.whl
!pip install -q /kaggle/input/pycocotools-206/wheels/pycocotools-2.0.6-cp310-cp310-linux_x86_64.whl

!protoc --version

libprotoc 3.20.3


## Install tensorflow models for the OpenImages Chanllenge Evaluator
In order to use the evaluator, we need to run the `protoc` command for the proto files, which is not allowed in the read only kaggle input dataset. Therefore, we copy the input to the working directory.

For local test, you can just clone the `tensorflow models` repo from https://github.com/tensorflow/models and run the `protoc object_detection/protos/*.proto --python_out=.` in the `research` sub dir in your local repo. 

In [2]:
!cp -r  /kaggle/input/tf-models-v213 /kaggle/working/tf-models-v213
!cd /kaggle/working/tf-models-v213/research && protoc object_detection/protos/*.proto --python_out=.

# Import OpenImages Challenge Evaluator
For local test, update the `TF_MODEL_REPO_PATH` to your cloned repo path. This block should not throw any error if you have all the dependencies and finish the `protoc` command. 

In [3]:
import sys

# Change this to your cloned tensorflow models repo
TF_MODEL_REPO_PATH = "/kaggle/working/tf-models-v213/research"

sys.path.append(TF_MODEL_REPO_PATH)
from object_detection.utils.object_detection_evaluation import (
    OpenImagesInstanceSegmentationChallengeEvaluator,
)
from object_detection.metrics.oid_challenge_evaluation_utils import (
    build_groundtruth_dictionary,
    build_predictions_dictionary,
)

/opt/conda/lib/python3.10/site-packages/tensorflow_io/python/ops/__init__.py:98: UserWarning: unable to load libtensorflow_io_plugins.so: unable to open file: libtensorflow_io_plugins.so, from paths: ['/opt/conda/lib/python3.10/site-packages/tensorflow_io/python/ops/libtensorflow_io_plugins.so']
caused by: ['/opt/conda/lib/python3.10/site-packages/tensorflow_io/python/ops/libtensorflow_io_plugins.so: undefined symbol: _ZN3tsl6StatusC1EN10tensorflow5error4CodeESt17basic_string_viewIcSt11char_traitsIcEENS_14SourceLocationE']
  warnings.warn(f"unable to load libtensorflow_io_plugins.so: {e}")
/opt/conda/lib/python3.10/site-packages/tensorflow_io/python/ops/__init__.py:104: UserWarning: file system plugins are not loaded: unable to open file: libtensorflow_io.so, from paths: ['/opt/conda/lib/python3.10/site-packages/tensorflow_io/python/ops/libtensorflow_io.so']
caused by: ['/opt/conda/lib/python3.10/site-packages/tensorflow_io/python/ops/libtensorflow_io.so: undefined symbol: _ZTVN10tenso

In [4]:
from tqdm import tqdm
import pandas as pd
import cv2
import json

# Global Configs

In [5]:
# Class, label conversion directories. Probably don't need to change these unless you need to add unsure class.
OID_CATEGORIES = [{"id": 1, "name": "blood_vessel"}, {"id": 2, "name": "glomerulus"}]
OID_CLASS_LABEL_MAP = {"blood_vessel": 1, "glomerulus": 2}

SUBM_TO_CLASS_LABEL = {"0": "blood_vessel", "1": "glomerulus"}
GT_CATEGORIES = ["blood_vessel", "glomerulus"]


# The validation output generated by you inference pipeline in the submission format. I have provided a small sample from dataset 1 for testing.
# Update this to your output file.
VAL_SUBM_PATH = "/kaggle/input/hubmap-eval-ds1-sample-subm/sample_submission_ds1_val_fold1.csv"

# Input image directory and annoations from the competition dataset. Only need to be updated for local evaluation.
TRAIN_IMAGE_DIR = "/kaggle/input/hubmap-hacking-the-human-vasculature/train" 
POLYGON_JSON_PATH = "/kaggle/input/hubmap-hacking-the-human-vasculature/polygons.jsonl"

# Evaluation IOU thresholds. 0.5 and 0.75 are common in other evaluators and they are only used for checking the correctness. Use 0.6 for the competition evaluation. 
OID_IOUS = [0.5,0.6,0.75]
# OID_IOUS = [0.6]


## Data Utils

In [6]:
import base64
import numpy as np
from pycocotools import _mask as coco_mask
import typing as t
import zlib

def encode_binary_mask(mask: np.ndarray) -> t.Text:
  """Converts a binary mask into OID challenge encoding ascii text."""

  # check input mask --
  if mask.dtype != bool:
    raise ValueError(
        "encode_binary_mask expects a binary mask, received dtype == %s" %
        mask.dtype)

  mask = np.squeeze(mask)
  if len(mask.shape) != 2:
    raise ValueError(
        "encode_binary_mask expects a 2d mask, received shape == %s" %
        mask.shape)

  # convert input mask to expected COCO API input --
  mask_to_encode = mask.reshape(mask.shape[0], mask.shape[1], 1)
  mask_to_encode = mask_to_encode.astype(np.uint8)
  mask_to_encode = np.asfortranarray(mask_to_encode)

  # RLE encode mask --
  encoded_mask = coco_mask.encode(mask_to_encode)[0]["counts"]

  # compress and base64 encoding --
  binary_str = zlib.compress(encoded_mask, zlib.Z_BEST_COMPRESSION)
  base64_str = base64.b64encode(binary_str)
  return base64_str

# Utils for generating the ground truth in OpenImages format
Modified from: 
- https://www.kaggle.com/code/benihime91/hubmap-2023-create-coco-annotations
- https://www.kaggle.com/code/ammarnassanalhajali/hubmap-2023-k-fold-cv-coco-dataset-generator

In [7]:
def coordinates_to_masks(coordinates, shape):
    masks = []
    for coord in coordinates:
        mask = np.zeros(shape, dtype=np.uint8)
        cv2.fillPoly(mask, [np.array(coord)], 1)
        masks.append(mask)
    return masks

def binary_mask_to_rle(binary_mask):
    rle = {'counts': [], 'size': list(binary_mask.shape)}
    counts = rle.get('counts')
    for i, (value, elements) in enumerate(itertools.groupby(binary_mask.ravel(order='F'))):
        if i == 0 and value == 1:
            counts.append(0)
        counts.append(len(list(elements)))
    return rle


def generate_oid_gt_df(images_ids, polygon_json_path):
    polygons_data = []        
    
    num_val_images = len(images_ids)
    num_val_images_in_polygons = 0
    
    # Load polygons for the images in the validation list.
    with open(polygon_json_path, "r") as file:
        for line in file:
            item = json.loads(line)
            if item["id"] in images_ids:
                polygons_data.append(json.loads(line))
                num_val_images_in_polygons +=1
    assert num_val_images == num_val_images_in_polygons
    
    image_id_list = []
    label_name = []
    xmin = []
    xmax = []
    ymin = []
    ymax = []
    mask = []
    mask_size = []
    
    IMG_WIDTH = 512
    IMG_HEIGHT = 512

    for item in tqdm(polygons_data):
        image_id = item["id"]
        anns = item["annotations"]
        empty = True
        for an in anns:
            category_type = an["type"]
            if category_type in GT_CATEGORIES:
                empty = False
                segmentation = an["coordinates"]
                mask_img = coordinates_to_masks(segmentation, (IMG_HEIGHT, IMG_WIDTH))[0]
                ys, xs = np.where(mask_img)
                x1, x2 = min(xs), max(xs)
                y1, y2 = min(ys), max(ys)

                mask_img = mask_img.astype(bool)
                mask_size.append(mask_img.sum())
                encoded_mask = encode_binary_mask(mask_img).decode()

                image_id_list.append(image_id)
                label_name.append(category_type)
                xmin.append(int(x1) / IMG_WIDTH)
                xmax.append(int(x2) / IMG_WIDTH)
                ymin.append(int(y1) / IMG_HEIGHT)
                ymax.append(int(y2) / IMG_HEIGHT)
                mask.append(encoded_mask)

        # Make sure empty images are also added to the ground truth dataset for false positive evaluation.
        if empty:
            image_id_list.append(image_id)
            label_name.append(categories_list[0])
            xmin.append(None)
            xmax.append(None)
            ymin.append(None)
            ymax.append(None)
            mask.append(None)

    num = len(image_id_list)
    is_group_of = [0] * num
    confidence = [1] * num
    image_width = [IMG_WIDTH] * num
    image_height = [IMG_HEIGHT] * num

    df_data = {
        "ImageID": image_id_list,
        "LabelName": label_name,
        "XMin": xmin,
        "XMax": xmax,
        "YMin": ymin,
        "YMax": ymax,
        "IsGroupOf": is_group_of,
        "Confidence": confidence,
        "ImageWidth": image_width,
        "ImageHeight": image_height,
        "Mask": mask,
    }
    return pd.DataFrame(df_data)

## Convert the submission file to the format for OpenImages Challenge Evaluation

In [8]:
def subm_to_pred_df(subm_df):
    ids = []
    label = []
    image_width = []
    image_height = []
    score = []
    mask = []
    for index, row in subm_df.iterrows():
        pred_strs = row["prediction_string"].split(" ")
        labels_str = pred_strs[0::3]
        scores = pred_strs[1::3]
        img_masks = pred_strs[2::3]
        assert len(scores) == len(img_masks)
        for i in range(len(scores)):
            ids.append(row["id"])
            label.append(SUBM_TO_CLASS_LABEL[labels_str[i]])
            image_width.append(row["width"])
            image_height.append(row["height"])
            score.append(scores[i])
            mask.append(img_masks[i])

    pred_df = pd.DataFrame(
        {
            "ImageID": ids,
            "LabelName": label,
            "ImageWidth": image_width,
            "ImageHeight": image_height,
            "Score": score,
            "Mask": mask,
        }
    )

    return pred_df

## Evaluation function 
Modified from: https://github.com/tensorflow/models/blob/master/research/object_detection/metrics/oid_challenge_evaluation.py

In [9]:
def get_oid_metrics(gt_df, pred_df, iou_thresholds=[0.5, 0.6, 0.75]):
    # print the number of imageID in gt_df and pred_df
    print("Number gt_df imageID: ", len(gt_df.groupby("ImageID")))
    print("Number of pred_df imageID: ", len(pred_df.groupby("ImageID")))

    gt_df = gt_df.loc[gt_df["ImageID"].isin(pred_df["ImageID"].unique())]
    gt_df.rename(columns={"Confidence": "ConfidenceImageLabel"}, inplace=True)

    gt_dicts = {}
    pred_dicts = {}
    print("Building groundtruth and prediction dictionary for oid evaluator")
    for _, groundtruth in tqdm(
        enumerate(gt_df.groupby("ImageID")), total=len(gt_df.groupby("ImageID"))
    ):
        image_id, image_groundtruth = groundtruth
        groundtruth_dictionary = build_groundtruth_dictionary(
            image_groundtruth, OID_CLASS_LABEL_MAP
        )
        gt_dicts[image_id] = groundtruth_dictionary
        prediction_dictionary = build_predictions_dictionary(
            pred_df.loc[pred_df["ImageID"] == image_id], OID_CLASS_LABEL_MAP
        )
        pred_dicts[image_id] = prediction_dictionary

    print("Calculating metrics for different iou_thresholds")
    metrics_list = []
    for iou_threshold in iou_thresholds:
        challenge_evaluator = OpenImagesInstanceSegmentationChallengeEvaluator(
            OID_CATEGORIES, matching_iou_threshold=iou_threshold
        )
        print("Iou_threshold: ", iou_threshold)
        for _, groundtruth in tqdm(
            enumerate(gt_df.groupby("ImageID")), total=len(gt_df.groupby("ImageID"))
        ):
            image_id, image_groundtruth = groundtruth

            challenge_evaluator.add_single_ground_truth_image_info(
                image_id, gt_dicts[image_id]
            )
            image_id, image_groundtruth = groundtruth
            challenge_evaluator.add_single_detected_image_info(
                image_id, pred_dicts[image_id]
            )

        metrics = challenge_evaluator.evaluate()
        metrics_list.append(metrics)
    return metrics_list

# Main

In [10]:
# Read the inference result for your validation set in the competition submission format
val_subm_df = pd.read_csv(VAL_SUBM_PATH) 
val_subm_df.head()

,id,height,width,prediction_string
0,0033bbc76b6b,512,512,0 0.04767665 eNrLiw8xNcqxN/I39ANiCISx/MCiCHFkC...
1,0754412b2917,512,512,0 0.1809986 eNrLz88wNcy3NwCBgABrg/x8F0MDOPA39D...
2,0788fc3be62e,512,512,0 0.83666736 eNoLi8kyNU+1N/OwcrfwMvMy8TX2MfE18...
3,07bdbe578ded,512,512,0 0.9291403 eNoLzE8xM860t/Ay8TX0N/Iz8jM0MPAHsQ...
4,13aa34ced90d,512,512,0 0.06446552 eNrLj88wMcyzNzQw8AdiPyN/MAtEGvobg...


In [11]:
# Get the image list from the validation output
val_image_ids = val_subm_df["id"].to_list()
print(len(val_image_ids))

85


In [12]:
# Generate the ground truth dataframe in OpenImages Challenge format for the validation images.
# You can also dump this to a csv file and reload it to avoid running the generation function again.
gt_df = generate_oid_gt_df(val_image_ids, POLYGON_JSON_PATH)
gt_df.head()

100%|██████████| 85/85 [00:02<00:00, 34.05it/s]


,ImageID,LabelName,XMin,XMax,YMin,YMax,IsGroupOf,Confidence,ImageWidth,ImageHeight,Mask
0,0033bbc76b6b,blood_vessel,0.279297,0.353516,0.269531,0.445312,0,1,512,512,eNqLSg03SjCIs7eJ87fyMfUy8zH2MfYz8jXyNwax/Yz8TL...
1,0033bbc76b6b,blood_vessel,0.000000,0.011719,0.054688,0.115234,0,1,512,512,eNrLNcg3CLA3NPA38jHxM/Ex8cqPyzEHAD/KBeo=
2,0033bbc76b6b,blood_vessel,0.726562,0.794922,0.968750,0.998047,0,1,512,512,eNrLj88yNcyxN/Yz8jeEQAMDEAaREDZCxNAfhEEkgocQNf...
3,0754412b2917,blood_vessel,0.726562,0.787109,0.617188,0.833984,0,1,512,512,eNoVycsOgjAQQNFf6hRl4RQJMCukgwpiIhV8VEAo+v87cX...
4,0754412b2917,blood_vessel,0.085938,0.126953,0.466797,0.539062,0,1,512,512,eNpLCk8zMMm2N/Y18jP2A5L+hokOPvl2yQaG/gb+RkBs7G...


In [13]:
# Convert the validation output to OpenImages Challenge submission format
pred_df = subm_to_pred_df(val_subm_df)
pred_df.head()

,ImageID,LabelName,ImageWidth,ImageHeight,Score,Mask
0,0033bbc76b6b,blood_vessel,512,512,0.04767665,eNrLiw8xNcqxN/I39ANiCISx/MCiCHFkCBM1MABhEDD0h2...
1,0033bbc76b6b,blood_vessel,512,512,0.7571677,eNoLsg42zLUzMDAw9DP0N/I38DPyN/Iz8jX2NvU18cwOyT...
2,0033bbc76b6b,blood_vessel,512,512,0.016928185,eNoLCogyNM62N/Ez9Df2NfQHw2QHpzB7u3QH93D7RAMDQ3...
3,0033bbc76b6b,blood_vessel,512,512,0.06374227,eNoLSEgyMMy3N/Q3MDD0N/IDYRA09AeJgMRApL8hBCNEIK...
4,0033bbc76b6b,blood_vessel,512,512,0.030350031,eNoLSEg2MMqzN/JDQEN/Q38DAwgJohEkDPgbwmhUCJHBYB...


In [14]:
# Calculate OpenImages metrics.
metrics_list = get_oid_metrics(gt_df, pred_df, iou_thresholds=OID_IOUS)

Number gt_df imageID:  85
Number of pred_df imageID:  85
Building groundtruth and prediction dictionary for oid evaluator


100%|██████████| 85/85 [00:35<00:00,  2.40it/s]


Calculating metrics for different iou_thresholds
Iou_threshold:  0.5


100%|██████████| 85/85 [00:18<00:00,  4.52it/s]


Iou_threshold:  0.6


100%|██████████| 85/85 [00:18<00:00,  4.53it/s]


Iou_threshold:  0.75


100%|██████████| 85/85 [00:18<00:00,  4.49it/s]


In [15]:
# Print the metrics
for metrics in metrics_list:
    for item in metrics:
        print("--", item, metrics[item])
    print(" ")

-- OpenImagesInstanceSegmentationChallenge_Precision/mAP@0.5IOU 0.3411267599635575
-- OpenImagesInstanceSegmentationChallenge_PerformanceByCategory/AP@0.5IOU/blood_vessel 0.682253519927115
-- OpenImagesInstanceSegmentationChallenge_PerformanceByCategory/AP@0.5IOU/glomerulus 0.0
 
-- OpenImagesInstanceSegmentationChallenge_Precision/mAP@0.6IOU 0.2909974285533541
-- OpenImagesInstanceSegmentationChallenge_PerformanceByCategory/AP@0.6IOU/blood_vessel 0.5819948571067082
-- OpenImagesInstanceSegmentationChallenge_PerformanceByCategory/AP@0.6IOU/glomerulus 0.0
 
-- OpenImagesInstanceSegmentationChallenge_Precision/mAP@0.75IOU 0.12145058973617051
-- OpenImagesInstanceSegmentationChallenge_PerformanceByCategory/AP@0.75IOU/blood_vessel 0.24290117947234102
-- OpenImagesInstanceSegmentationChallenge_PerformanceByCategory/AP@0.75IOU/glomerulus 0.0
 


# Notes

## Validation of results
- I compared the results using the COCO `mAP_50` and `mAP_75` calculated from mmdetection and there are small differences.
    - 0.681 -> 0.6823
    - 0.245 -> 0.2429
- See this discussion for more info about the difference: https://www.kaggle.com/competitions/hubmap-hacking-the-human-vasculature/discussion/419309#2316979

- The mmdetection COCO metrics for the model:

```
+--------------+-------+--------+--------+-------+-------+-------+
| category     | mAP   | mAP_50 | mAP_75 | mAP_s | mAP_m | mAP_l |
+--------------+-------+--------+--------+-------+-------+-------+
| blood_vessel | 0.319 | 0.681  | 0.245  | 0.261 | 0.397 | 0.525 |
+--------------+-------+--------+--------+-------+-------+-------+
```

- OpenImages Metrics calculated from this notebook:

```
-- OpenImagesInstanceSegmentationChallenge_Precision/mAP@0.5IOU 0.3411267599635575
-- OpenImagesInstanceSegmentationChallenge_PerformanceByCategory/AP@0.5IOU/blood_vessel 0.682253519927115
-- OpenImagesInstanceSegmentationChallenge_PerformanceByCategory/AP@0.5IOU/glomerulus 0.0
 
-- OpenImagesInstanceSegmentationChallenge_Precision/mAP@0.6IOU 0.2909974285533541
-- OpenImagesInstanceSegmentationChallenge_PerformanceByCategory/AP@0.6IOU/blood_vessel 0.5819948571067082
-- OpenImagesInstanceSegmentationChallenge_PerformanceByCategory/AP@0.6IOU/glomerulus 0.0
 
-- OpenImagesInstanceSegmentationChallenge_Precision/mAP@0.75IOU 0.12145058973617051
-- OpenImagesInstanceSegmentationChallenge_PerformanceByCategory/AP@0.75IOU/blood_vessel 0.24290117947234102
-- OpenImagesInstanceSegmentationChallenge_PerformanceByCategory/AP@0.75IOU/glomerulus 0.0
```
- **Be careful about overfitting to WSI1 and WSI2.**
- You may want to control the number of validation images(<500) since the metrics calculation requires some amount of memory. It should be totally fine with the 422 images from dataset 1.



### Let me know if there is a mistake in this notebook.